<a href="https://colab.research.google.com/github/emilymhudson/soc-command-center/blob/main/Security_Health_Score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install PyGithub

In [ ]:
%%writefile orchestrator.py
import os
from github import Github

def run_audit():
    # Retrieve token securely from the environment
    token = os.getenv('GITHUB_TOKEN')

    if not token:
        print("[!] ERROR: GITHUB_TOKEN environment variable not found.")
        return

    g = Github(token)
    user = g.get_user()

    repos_to_audit = [
        "probabilistic-threat-classifier",
        "mime-forensic-log-parser",
        "detection-as-code-hub",
        "academic-ctf-architecture",
        "evidentiary-provenance-engine"
    ]

    report = "# SOC Command Center: Security Posture Dashboard\n\n"
    report += f"Generated: {os.popen('date').read()}\n\n| Repository | Status | Health Score |\n|---|---|---|\n"

    total_score = 0
    for repo_name in repos_to_audit:
        try:
            repo = user.get_repo(repo_name)
            status = "Operational"
            score = 20
        except Exception:
            status = "NOT FOUND"
            score = 0

        report += f"| {repo_name} | {status} | {score}/20 |\n"
        total_score += score

    report += f"\n## Aggregate Security Posture: {total_score}/100"

    with open("Security_Posture_Report.md", "w") as f:
        f.write(report)
    print("[+] Security_Posture_Report.md generated successfully.")

if __name__ == "__main__":
    run_audit()

In [ ]:
import os
import getpass

# Securely set the token for this session
os.environ['GITHUB_TOKEN'] = getpass.getpass('Enter your GitHub Personal Access Token: ')

# 2. Run the audit
!python orchestrator.py

# 3. Sync with GitHub
!git config --global user.email "your-email@example.com"
!git config --global user.name "Your Name"

# If you haven't set the remote yet, do it once:
# !git remote add origin https://<YOUR_TOKEN>@github.com/<USERNAME>/soc-command-center.git

!git add Security_Posture_Report.md orchestrator.py
!git commit -m "Update SOC Dashboard and Orchestrator logic"
!git push origin main